# Modelling items export: load and reconcile

This notebook inspects the reusable modelling-items export. Cohort, joins, labels, and business rules live in `src/hidden_dissatisfaction`/dbt; this notebook only loads the published Parquet and manifest, validates the export contract, and reports compact quality summaries.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from hidden_dissatisfaction.application.modelling_export import (
    ModellingExportContract,
    validate_item_records,
)

EXPORT = ROOT / 'data' / 'processed' / 'modelling_items.parquet'
MANIFEST = EXPORT.with_suffix(EXPORT.suffix + '.manifest.json')
contract = ModellingExportContract()

In [ ]:
if not EXPORT.exists() or not MANIFEST.exists():
    raise FileNotFoundError(f'Missing export or sidecar: {EXPORT} / {MANIFEST}')

items = pd.read_parquet(EXPORT)
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
validate_item_records(items.to_dict('records'), contract)

actual_schema = {column: str(dtype) for column, dtype in items.dtypes.items()}
actual_nulls = {column: int(value) for column, value in items.isna().sum().items()}
assert manifest['row_count'] == len(items)
assert manifest['order_count'] == items[contract.order_key].nunique(dropna=False)
assert manifest['schema'] == actual_schema
assert manifest['null_profile'] == actual_nulls

schema_summary = pd.DataFrame({'dtype': items.dtypes.astype(str), 'nulls': items.isna().sum()})
count_summary = pd.Series({
    'rows': len(items),
    'distinct_items': items[contract.item_key].nunique(),
    'distinct_orders': items[contract.order_key].nunique(),
}, name='count')
display(count_summary.to_frame())
display(schema_summary)
display(pd.Series(manifest, name='manifest').to_frame())

## Optional dbt key reconciliation

Set `DBT_DUCKDB_PATH` (and optionally `DBT_SAMPLE_SIZE`) when a built dbt DuckDB relation is available. The hook compares a deterministic sample of `product_item_id` keys only; it does not rebuild or duplicate dbt logic.

In [ ]:
dbt_path = os.getenv('DBT_DUCKDB_PATH')
if dbt_path:
    import duckdb

    sample_size = int(os.getenv('DBT_SAMPLE_SIZE', '100'))
    sample = items[[contract.item_key]].sample(
        n=min(sample_size, len(items)), random_state=0
    )
    con = duckdb.connect(dbt_path, read_only=True)
    try:
        relation = os.getenv('DBT_RELATION', 'gold.ml_modelling_items')
        con.register('export_key_sample', sample)
        dbt_keys = con.execute(
            f'SELECT d.product_item_id FROM {relation} d JOIN export_key_sample s USING (product_item_id)'
        ).fetch_df()
    finally:
        con.close()
    missing = sorted(set(sample[contract.item_key]) - set(dbt_keys['product_item_id']))
    reconciliation = pd.Series({'sampled_keys': len(sample), 'dbt_matches': len(dbt_keys), 'missing_keys': len(missing)})
    display(reconciliation.to_frame('value'))
    assert not missing, f'Modelling export keys missing from dbt output: {missing[:5]}'
else:
    print('DBT_DUCKDB_PATH not supplied; skipped dbt key reconciliation.')